# Comment Classification: Pretrained Model

<!-- describe general idea of what we are doing here -->


## Setting up the Virtual Environment

To install Python libraries, navigate to the folder "like_dislike" in your terminal, then run `python3 -m venv .venv` to create a new virtual environment and activate it using `source .venv/bin/activate`. Run `pip3 install -r requirements.txt` to install required Python libraries into the created virtual environment.


## Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from transformers import pipeline
import torch
from scipy.stats import spearmanr
from sklearn.metrics import (classification_report, roc_auc_score, cohen_kappa_score,
                             f1_score, accuracy_score, mean_absolute_error)

## Data-Wrangling

Next, we load the comment data.

In [ ]:
# loading comments data
data_path = "data/comments/comments_manual_coding.xlsx"
data_comments = pd.read_excel(data_path)

## Hugging Face

Hugging Face provides an open-source platform that allows data scientists, computer scientists, and researchers to share models and datasets. We will use a model from its comprehensive transformer library. To do so:

1. Create an access token on [Hugging Face](https://huggingface.co/settings/tokens).

2. Run in terminal: `echo 'export HF_TOKEN="hf_yournewtoken"' >> ~/.zshrc && source ~/.zshrc`

3. Run to verify: `echo $HF_TOKEN`

In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## Codebook as NLI hypotheses

The coding scheme (`system_prompt.txt`) is broken down into yes/no statements that the zero-shot model checks one by one. The model gives each statement an entailment probability, and those probabilities are combined as the codebook prescribes:

| variable | codebook | aggregation |
|---|---|---|
| `pers_exp`, `emot_exp`, `pol_opin`, `contr` | 0/1, default 0 | 1 if **any** trigger hypothesis ≥ threshold |
| `breadth` | 0–3, number of topics from the 9-topic list, capped at 3 | number of topic hypotheses ≥ threshold, capped at 3 |
| `valence` | 1 positive, 2 negative, 3 ambivalent, 4 indifferent (**nominal**) | pos & neg → 3, only pos → 1, only neg → 2, neither → 4 |

This is a baseline. Rules that need context or fine judgement (parent comments, "zwar … aber", the plan-as-object exclusion) are beyond what zero-shot NLI can do.

In [ ]:
TEXT_COL = "raw"
MODEL = "MoritzLaurer/xlm-v-base-mnli-xnli"

CODEBOOK = {
    "pers_exp": {"type": "binary", "threshold": 0.5, "hypotheses": [
        "Der Autor berichtet von einer eigenen Erfahrung oder aus seinem eigenen Leben.",
        "Der Autor beschreibt, wie er die Terrorgefahr in seinem eigenen Alltag wahrnimmt.",
    ]},
    "emot_exp": {"type": "binary", "threshold": 0.5, "hypotheses": [
        "Der Autor drückt eigene Gefühle aus, zum Beispiel Angst, Sorge, Wut oder Trauer.",
        "Der Autor ist sichtlich aufgebracht und macht seinem Ärger Luft.",
    ]},
    "pol_opin": {"type": "binary", "threshold": 0.5, "hypotheses": [
        "Der Kommentar bewertet Politiker, Parteien oder die Regierung.",
        "Der Kommentar vertritt eine ideologische Position, etwa zu Migration oder Pazifismus.",
        "Der Kommentar fordert Massnahmen wie Grenzschliessung, Abschiebungen oder einen Stopp von Waffenexporten.",
    ]},
    "contr": {"type": "binary", "threshold": 0.5, "hypotheses": [
        "Der Kommentar stellt Migranten, Flüchtlinge, Religion oder Ethnie als Bedrohung dar.",
        "Der Kommentar fordert Grenzschliessung, Einreiseverbote oder Massenabschiebungen.",
        "Der Kommentar beschimpft Politiker oder Parteien pauschal.",
        "Der Kommentar vertritt eine radikale politische Position.",
    ]},
    "breadth": {"type": "count", "cap": 3, "threshold": 0.5, "hypotheses": [
        "Der Kommentar kritisiert das Versagen von Politikern oder politischen Institutionen.",
        "Der Kommentar spricht über Migration, Flüchtlinge oder Integration.",
        "Der Kommentar spricht über Religion.",
        "Der Kommentar spricht über Sicherheitsmassnahmen wie Polizei, Überwachung oder Kontrollen.",
        "Der Kommentar spricht über Grundrechte, Freiheitsrechte oder Datenschutz.",
        "Der Kommentar spricht über internationale Zusammenarbeit, etwa zwischen Geheimdiensten.",
        "Der Kommentar spricht über Ungerechtigkeit, soziale Missstände oder Leid.",
        "Der Kommentar spricht über Terroranschläge, Terrorismus oder Radikalisierung.",
        "Der Kommentar spricht über Demokratie, Wahlen, Parteien, Bildung oder politische Teilhabe.",
    ]},
    "valence": {"type": "valence", "threshold": 0.5, "hypotheses": {
        "pos": "Der Autor bewertet etwas ausdrücklich positiv, stimmt zu oder äussert Hoffnung.",
        "neg": "Der Autor äussert Kritik, Ablehnung, Zweifel oder Pessimismus.",
    }},
}
VALENCE_NAMES = {1: "positive", 2: "negative", 3: "ambivalent", 4: "indifferent"}

def hyps(spec):
    h = spec["hypotheses"]
    return list(h.values()) if isinstance(h, dict) else h

ALL_HYPS = list(dict.fromkeys(h for spec in CODEBOOK.values() for h in hyps(spec)))

# codebook: 4+ topics are coded as 3
n_over = int((data_comments["breadth"] > 3).sum())
if n_over:
    print(f"breadth: {n_over} manual codes > 3, capped at 3 per codebook")
data_comments["breadth_manual_raw"] = data_comments["breadth"]
data_comments["breadth"] = data_comments["breadth"].clip(upper=3)

# sanity check: manual codes must be valid under the codebook
ALLOWED = {"binary": {0, 1}, "count": {0, 1, 2, 3}, "valence": {1, 2, 3, 4}}
for var, spec in CODEBOOK.items():
    assert var in data_comments.columns, f"{var} not in data"
    bad = set(data_comments[var].dropna().unique()) - ALLOWED[spec["type"]]
    assert not bad, f"{var}: invalid manual codes {bad}"

## Classification

In [ ]:
device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else -1)
clf = pipeline("zero-shot-classification", model=MODEL, device=device)

# codebook: quoted text is not the commenter's own words
QUOTE_RE = re.compile(r"\[quote[^\]]*\].*?\[/quote\]", flags=re.S | re.I)
texts = (data_comments[TEXT_COL].fillna("").astype(str)
         .str.replace(QUOTE_RE, " ", regex=True).str.strip())
valid = texts != ""

# one pass: every comment x every hypothesis, independent entailment probabilities
out = clf(texts[valid].tolist(), candidate_labels=ALL_HYPS, hypothesis_template="{}",
          multi_label=True, batch_size=16)
out = [out] if isinstance(out, dict) else out

H = pd.DataFrame(np.nan, index=data_comments.index, columns=ALL_HYPS)  # empty text -> NaN
H.loc[valid] = [[dict(zip(r["labels"], r["scores"]))[h] for h in ALL_HYPS] for r in out]

In [ ]:
def predict(var, thr=None):
    """Aggregate hypothesis scores to the codebook variable. NaN (empty text) -> False."""
    spec = CODEBOOK[var]
    thr = spec["threshold"] if thr is None else thr
    if spec["type"] == "binary":
        return (H[hyps(spec)].max(axis=1) >= thr).astype(int)
    if spec["type"] == "count":
        return (H[hyps(spec)] >= thr).sum(axis=1).clip(upper=spec["cap"]).astype(int)
    pos = H[spec["hypotheses"]["pos"]] >= thr
    neg = H[spec["hypotheses"]["neg"]] >= thr
    return pd.Series(np.select([pos & neg, pos, neg], [3, 1, 2], 4), index=H.index)

for var, spec in CODEBOOK.items():
    data_comments[f"{var}_pred"] = predict(var)
    if spec["type"] == "binary":
        data_comments[f"{var}_score"] = H[hyps(spec)].max(axis=1)        # strongest trigger
    elif spec["type"] == "count":
        data_comments[f"{var}_soft"] = H[hyps(spec)].sum(axis=1).clip(upper=spec["cap"])  # expected no. of topics
    else:
        data_comments["valence_pos_score"] = H[spec["hypotheses"]["pos"]]
        data_comments["valence_neg_score"] = H[spec["hypotheses"]["neg"]]

## Evaluation against the manual coding

- **binary**: precision/recall/F1, Cohen's κ, ROC AUC (threshold-free).
- **breadth** (ordinal): quadratic-weighted κ, MAE, share within ±1, Spearman ρ with the soft topic count.
- **valence** (nominal, 1–4 are categories, not a scale): per-class report, macro F1, unweighted κ.

With rare categories, look at F1 and κ rather than accuracy.

In [ ]:
summary = []
for var, spec in CODEBOOK.items():
    gold = data_comments[var].astype(int)
    pred = data_comments[f"{var}_pred"]
    print(f"\n===== {var} =====")
    print(pd.crosstab(gold, pred, rownames=["manual"], colnames=["model"]))

    row = {"variable": var, "type": spec["type"],
           "accuracy": accuracy_score(gold, pred),
           "macro_F1": f1_score(gold, pred, average="macro", zero_division=0),
           "kappa": cohen_kappa_score(gold, pred)}

    if spec["type"] == "binary":
        print(classification_report(gold, pred, digits=3, zero_division=0))
        score = data_comments[f"{var}_score"].fillna(0)
        row.update({"prevalence": gold.mean(),
                    "F1_pos": f1_score(gold, pred, zero_division=0),
                    "AUC": roc_auc_score(gold, score) if gold.nunique() == 2 else np.nan})
    elif spec["type"] == "count":
        row.update({"kappa_quadratic": cohen_kappa_score(gold, pred, weights="quadratic"),
                    "MAE": mean_absolute_error(gold, pred),
                    "within_1": (abs(gold - pred) <= 1).mean(),
                    "spearman": spearmanr(gold, data_comments[f"{var}_soft"].fillna(0))[0]})
    else:
        labels = sorted(VALENCE_NAMES)
        print(classification_report(gold, pred, labels=labels,
                                    target_names=[VALENCE_NAMES[l] for l in labels],
                                    digits=3, zero_division=0))
    summary.append(row)

summary = pd.DataFrame(summary).set_index("variable").round(3)
summary

### Threshold diagnostic

For each variable, the threshold that maximises its main metric (binary: F1 of class 1; breadth: quadratic κ; valence: macro F1). It is chosen **in-sample** on the same 150 comments, so the scores are optimistic. Use it to set `threshold` in `CODEBOOK`, then validate on new hand-coded comments or with cross-validation.

In [ ]:
def main_metric(var, pred):
    gold = data_comments[var].astype(int)
    t = CODEBOOK[var]["type"]
    if t == "binary":
        return f1_score(gold, pred, zero_division=0)
    if t == "count":
        return cohen_kappa_score(gold, pred, weights="quadratic")
    return f1_score(gold, pred, average="macro", zero_division=0)

grid = np.round(np.arange(0.05, 0.96, 0.05), 2)
for var, spec in CODEBOOK.items():
    res = pd.Series({t: main_metric(var, predict(var, t)) for t in grid})
    print(f"{var:9s} best threshold = {res.idxmax():.2f} -> {res.max():.3f}   "
          f"(at {spec['threshold']}: {main_metric(var, predict(var)):.3f})")

In [ ]:
data_comments.to_excel("data/comments/comments_model_coding.xlsx", index=False)